# AgroAdapt Ebéjico — Datos geoespaciales de la NASA
**Integrante 1 · GIS / NASA**

Este cuaderno muestra, paso a paso, lo que dicen los satélites y las bases de datos de la NASA sobre el municipio de **Ebéjico (Antioquia)**:

1. ¿Dónde está y qué tan grande es el municipio?
2. Cómo se ve desde el espacio (**Sentinel-2** y **Landsat**)
3. Qué tan verde y húmeda está la vegetación
4. El relieve: alturas y pendientes (**NASADEM / SRTM**)
5. El clima: temperatura, lluvia y humedad del suelo (**NASA POWER**)

**Ningún número está escrito a mano**: todo se consulta en línea en el momento de ejecutar el cuaderno.
Para correrlo: menú **Run All** (Ejecutar todo). Tarda alrededor de 1–2 minutos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datos_nasa as dn

# Periodo que queremos revisar (se puede cambiar)
INICIO, FIN = "2025-11-01", "2026-11-15"

pd.set_option("display.precision", 2)
plt.rcParams["figure.dpi"] = 100

## 1. ¿Dónde está Ebéjico?
El límite oficial del municipio se descarga de **geoBoundaries** (base de datos abierta de límites administrativos).
Con ese contorno se recortan todas las demás capas.

In [ ]:
limite = dn.limite_municipio("Ebéjico")
centro = limite.to_crs("EPSG:32618").centroid.to_crs("EPSG:4326").iloc[0]
LAT, LON = round(centro.y, 4), round(centro.x, 4)
AREA_KM2 = limite.to_crs("EPSG:32618").area.iloc[0] / 1e6

ax = limite.plot(figsize=(5, 6), color="#d9f0d3", edgecolor="darkgreen", linewidth=2)
ax.plot(LON, LAT, "r*", markersize=15)
ax.set_title("Límite municipal de Ebéjico (★ = centro)")
plt.show()

pd.DataFrame({"Dato": ["Municipio", "Área (km²)", "Latitud del centro", "Longitud del centro"],
              "Valor": [limite["shapeName"].iloc[0], round(AREA_KM2, 1), LAT, LON]})

## 2. Ebéjico visto desde el espacio — Sentinel-2
**Sentinel-2** toma una foto de la zona cada ~5 días con píxeles de 10 m.
Ebéjico es muy nublado, así que primero buscamos las fechas más despejadas.
El porcentaje de nubes **sobre Ebéjico** lo calcula el propio producto satelital (capa de clasificación de escena).

In [ ]:
s2 = dn.buscar_escenas(limite, "sentinel-2-l2a", INICIO, FIN, max_nubes=50)
print(f"Sentinel-2 pasó {len(s2)} veces sobre Ebéjico entre {INICIO} y {FIN} (con menos de 50% de nubes en la escena).")

s2 = s2.nsmallest(15, "nubes_escena_pct").copy()
s2[["cubre_ebejico_%", "nubes_sobre_ebejico_%"]] = [dn.calidad_escena(limite, "sentinel-2-l2a", ids) for ids in s2["ids"]]
# Primero las tomas que cubren casi todo el municipio; entre ellas, las de menos nubes
s2 = s2[s2["cubre_ebejico_%"] >= 95].sort_values("nubes_sobre_ebejico_%")
s2[["fecha", "satelite", "cubre_ebejico_%", "nubes_sobre_ebejico_%"]]

In [ ]:
# Índices calculados por el servidor (el "-2000" corrige el desfase de +1000 que trae cada banda del producto)
NDVI_S2 = "(B08-B04)/(B08+B04-2000)"   # verdor de la vegetación
NDMI_S2 = "(B08-B11)/(B08+B11-2000)"   # agua dentro de las hojas

mejor = s2.iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
dn.mostrar(dn.imagen(limite, "sentinel-2-l2a", mejor.ids, assets="visual"),
           f"Color real — Sentinel-2 — {mejor.fecha}", ax[0])
dn.mostrar(dn.imagen(limite, "sentinel-2-l2a", mejor.ids, expression=NDVI_S2, asset_as_band="true",
                     colormap_name="rdylgn", rescale="0,1"),
           f"Verdor de la vegetación (NDVI) — {mejor.fecha}", ax[1],
           leyenda="rojo = suelo, vías, poca vegetación   ·   verde oscuro = vegetación densa")
plt.show()

## 3. ¿Qué tan verde y húmeda está la vegetación?
- **NDVI** (verdor): va de 0 a 1. Cerca de 1 = vegetación densa y sana; cerca de 0 = suelo desnudo o construcciones.
- **NDMI** (humedad de las hojas): más alto = plantas con más agua; si baja con el tiempo puede indicar estrés por sequía.

Se usan las 5 fechas más despejadas. Las nubes que queden bajan un poco el promedio, por eso se muestra su porcentaje al lado.

In [ ]:
filas = []
for _, f in s2.head(5).iterrows():
    ndvi = dn.estadisticas(limite, "sentinel-2-l2a", f.ids, expression=NDVI_S2, asset_as_band="true")
    ndmi = dn.estadisticas(limite, "sentinel-2-l2a", f.ids, expression=NDMI_S2, asset_as_band="true")
    filas.append({"fecha": f.fecha, "satélite": "Sentinel-2", "nubes_%": f["nubes_sobre_ebejico_%"],
                  "verdor_NDVI": ndvi["media"], "humedad_hojas_NDMI": ndmi["media"]})
vegetacion = pd.DataFrame(filas).sort_values("fecha")
vegetacion

### Landsat 8 y 9
**Landsat** (NASA/USGS) pasa cada ~8 días con píxeles de 30 m. Además de los colores, tiene una **banda térmica**
que mide la **temperatura de la superficie** (suelo y hojas, no del aire).
Las fórmulas usan los factores de escala que publica el USGS para el producto Collection 2 Level-2.

In [ ]:
ls = dn.buscar_escenas(limite, "landsat-c2-l2", INICIO, FIN, max_nubes=60)
print(f"Landsat pasó {len(ls)} veces sobre Ebéjico (con menos de 60% de nubes en la escena).")
ls[["cubre_ebejico_%", "nubes_sobre_ebejico_%"]] = [dn.calidad_escena(limite, "landsat-c2-l2", ids) for ids in ls["ids"]]
ls = ls[ls["cubre_ebejico_%"] >= 95].sort_values("nubes_sobre_ebejico_%")
display(ls[["fecha", "satelite", "cubre_ebejico_%", "nubes_sobre_ebejico_%"]].head(6))

# reflectancia = DN * 0.0000275 - 0.2  ->  en la razón queda el término 2*0.2/0.0000275
NDVI_LS = "(nir08-red)/(nir08+red-14545.45)"
NDMI_LS = "(nir08-swir16)/(nir08+swir16-14545.45)"
TEMP_LS = "lwir11*0.00341802+149-273.15"      # Kelvin -> °C

filas = []
for _, f in ls.head(4).iterrows():
    filas.append({"fecha": f.fecha, "satélite": f.satelite, "nubes_%": f["nubes_sobre_ebejico_%"],
                  "verdor_NDVI": dn.estadisticas(limite, "landsat-c2-l2", f.ids, expression=NDVI_LS, asset_as_band="true")["media"],
                  "humedad_hojas_NDMI": dn.estadisticas(limite, "landsat-c2-l2", f.ids, expression=NDMI_LS, asset_as_band="true")["media"],
                  "temp_superficie_C": dn.estadisticas(limite, "landsat-c2-l2", f.ids, expression=TEMP_LS, asset_as_band="true")["media"]})
vegetacion = pd.concat([vegetacion, pd.DataFrame(filas)], ignore_index=True).sort_values("fecha")
vegetacion

In [ ]:
mejor_ls = ls.iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
dn.mostrar(dn.imagen(limite, "landsat-c2-l2", mejor_ls.ids, assets=["red", "green", "blue"], rescale="7000,14000"),
           f"Color real — {mejor_ls.satelite} — {mejor_ls.fecha}", ax[0])
t = dn.estadisticas(limite, "landsat-c2-l2", mejor_ls.ids, expression=TEMP_LS, asset_as_band="true")
dn.mostrar(dn.imagen(limite, "landsat-c2-l2", mejor_ls.ids, expression=TEMP_LS, asset_as_band="true",
                     colormap_name="coolwarm", rescale=f"{t['minimo']:.0f},{t['maximo']:.0f}"),
           f"Temperatura de la superficie — {mejor_ls.fecha}", ax[1],
           leyenda=f"azul = {t['minimo']:.0f} °C (zonas altas, bosque)   ·   rojo = {t['maximo']:.0f} °C (suelo expuesto, zonas bajas)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for sat, d in vegetacion.groupby("satélite"):
    ax.plot(pd.to_datetime(d["fecha"]), d["verdor_NDVI"], "o-", label=f"Verdor (NDVI) {sat}")
    ax.plot(pd.to_datetime(d["fecha"]), d["humedad_hojas_NDMI"], "s--", label=f"Humedad hojas (NDMI) {sat}")
ax.set_title("Vegetación de Ebéjico en el tiempo (promedio del municipio)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()

## 4. El relieve: alturas y pendientes
**NASADEM** es el modelo de elevación de la misión SRTM (transbordador espacial, NASA) reprocesado.
Con la elevación se calcula la **pendiente**: qué tan inclinado es el terreno (0% plano, 100% = 45°).
La pendiente es clave para saber dónde se puede mecanizar, dónde hay riesgo de erosión y qué cultivos son viables.

In [ ]:
dem, pendiente, extension, AREA_PIXEL = dn.relieve(limite)

fig, ax = plt.subplots(1, 2, figsize=(15, 7))
im = ax[0].imshow(dem, cmap="terrain", extent=extension)
fig.colorbar(im, ax=ax[0], label="metros sobre el nivel del mar"); ax[0].set_title("Elevación")
im = ax[1].imshow(pendiente, cmap="magma_r", vmin=0, vmax=np.nanpercentile(pendiente, 98), extent=extension)
fig.colorbar(im, ax=ax[1], label="% de pendiente"); ax[1].set_title("Pendiente del terreno")
for a in ax:
    limite.boundary.plot(ax=a, color="black", linewidth=1); a.set_axis_off()
plt.show()

pd.DataFrame({
    "Dato": ["Punto más bajo (m)", "Punto más alto (m)", "Altura promedio (m)",
             "Pendiente promedio (%)", "La mitad del municipio tiene pendiente mayor a (%)"],
    "Valor": [np.nanmin(dem), np.nanmax(dem), np.nanmean(dem), np.nanmean(pendiente), np.nanmedian(pendiente)],
}).round(0)

In [ ]:
# ¿Cuánta área hay en cada franja de altura? (franjas de 250 m)
franjas = pd.cut(pd.Series(dem[np.isfinite(dem)]), bins=np.arange(250, np.nanmax(dem) + 250, 250))
area_por_altura = (franjas.value_counts().sort_index() * AREA_PIXEL).round(1).rename("área_km2")
area_por_altura.index = [f"{int(i.left)}–{int(i.right)} m" for i in area_por_altura.index]
area_por_altura.plot.barh(figsize=(8, 5), color="sienna", title="Área de Ebéjico por franja de altura (km²)")
plt.show()

## 5. El clima — NASA POWER
**NASA POWER** reúne datos de satélites y modelos de la NASA pensados para agricultura.
Primero el **año típico** (promedio de muchos años, mes a mes) y luego cómo se comportaron los **últimos años**.
La descripción y las unidades de cada variable vienen directamente de la API.

In [ ]:
VARIABLES = {
    "T2M": "Temperatura media (°C)",
    "T2M_MAX": "Temperatura máxima (°C)",
    "T2M_MIN": "Temperatura mínima (°C)",
    "PRECTOTCORR_SUM": "Lluvia total del mes (mm)",
    "RH2M": "Humedad del aire (%)",
    "GWETTOP": "Humedad del suelo superficial (0 seco – 1 saturado)",
    "GWETROOT": "Humedad del suelo en raíces (0 seco – 1 saturado)",
    "ALLSKY_SFC_SW_DWN": "Radiación solar (MJ/m²/día)",
}
normal, info = dn.clima_power(LAT, LON, list(VARIABLES))
info["nombre_en_español"] = info.index.map(VARIABLES)
info

In [ ]:
MESES = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
ES = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
año_tipico = normal.loc[MESES].rename(index=dict(zip(MESES, ES)), columns=VARIABLES)
año_tipico.loc["AÑO"] = normal.loc["ANN"].rename(VARIABLES)
año_tipico.T.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
m = año_tipico.loc[ES]
ax[0].bar(ES, m["Lluvia total del mes (mm)"], color="steelblue", label="lluvia (mm)")
ax2 = ax[0].twinx()
ax2.plot(ES, m["Temperatura media (°C)"], "o-", color="firebrick", label="temperatura (°C)")
ax[0].set_ylabel("mm de lluvia"); ax2.set_ylabel("°C")
ax[0].set_title("Año típico en Ebéjico: lluvia y temperatura")
ax[1].plot(ES, m["Humedad del suelo superficial (0 seco – 1 saturado)"], "o-", label="suelo superficial")
ax[1].plot(ES, m["Humedad del suelo en raíces (0 seco – 1 saturado)"], "s-", label="zona de raíces")
ax[1].set_title("Año típico: humedad del suelo"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

In [ ]:
# Últimos años, mes a mes (datos mensuales de NASA POWER)
año_actual = pd.Timestamp(FIN).year
mensual, _ = dn.clima_power(LAT, LON, list(VARIABLES), tipo="monthly", inicio=año_actual - 3, fin=año_actual - 1)
mensual = mensual[~mensual.index.str.endswith("13")]          # la fila "13" es el total anual
mensual.index = pd.to_datetime(mensual.index, format="%Y%m")
mensual = mensual.rename(columns=VARIABLES)

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(mensual.index, mensual["Lluvia total del mes (mm)"], width=20, color="steelblue")
ax.set_title(f"Lluvia mes a mes {año_actual-3}–{año_actual-1} (NASA POWER)"); ax.set_ylabel("mm")
plt.show()
por_año = mensual.groupby(mensual.index.year).agg({
    "Temperatura media (°C)": "mean",
    "Lluvia total del mes (mm)": "sum",
    "Humedad del suelo en raíces (0 seco – 1 saturado)": "mean",
}).rename(columns={"Lluvia total del mes (mm)": "Lluvia del año (mm)"})
por_año.index.name = "año"
por_año.round(2)

## 6. Resumen en palabras
Este texto se arma solo con los valores obtenidos arriba.

In [ ]:
v = vegetacion.dropna(subset=["verdor_NDVI"])
NOMBRE_MES = dict(zip(ES, ["enero","febrero","marzo","abril","mayo","junio","julio",
                           "agosto","septiembre","octubre","noviembre","diciembre"]))
mas_lluvioso = NOMBRE_MES[m["Lluvia total del mes (mm)"].idxmax()]
mas_seco = NOMBRE_MES[m["Lluvia total del mes (mm)"].idxmin()]
print(f'''
• Ebéjico tiene {AREA_KM2:.0f} km² y su terreno va de {np.nanmin(dem):.0f} a {np.nanmax(dem):.0f} m de altura.
• Es un municipio de montaña: la mitad del territorio tiene pendientes mayores a {np.nanmedian(pendiente):.0f}%.
• Según los satélites, el verdor promedio (NDVI) estuvo entre {v["verdor_NDVI"].min():.2f} y {v["verdor_NDVI"].max():.2f}
  (en una escala de 0 a 1), lo que indica un territorio mayormente cubierto de vegetación.
• La temperatura media del aire (NASA POWER) es de {año_tipico.loc["AÑO", "Temperatura media (°C)"]:.1f} °C
  y llueven unos {año_tipico.loc["AÑO", "Lluvia total del mes (mm)"]:.0f} mm al año.
• El mes más lluvioso suele ser {mas_lluvioso} y el más seco {mas_seco}.
• La humedad del suelo en la zona de raíces se mantiene en promedio en {año_tipico.loc["AÑO", "Humedad del suelo en raíces (0 seco – 1 saturado)"]:.2f}
  (0 = seco, 1 = saturado).
''')

In [ ]:
# Guardar las tablas para el resto del equipo (carpeta data/processed)
vegetacion.to_csv(dn.CARPETA / "vegetacion_satelites.csv", index=False)
año_tipico.to_csv(dn.CARPETA / "clima_año_tipico_power.csv")
mensual.to_csv(dn.CARPETA / "clima_mensual_power.csv")
print("Tablas guardadas en", dn.CARPETA)